# Docling PDF Parsing — Google Colab Version (Tesseract OCR)

Is version mein Tesseract OCR use ho raha hai (jaisa original notebook mein tha).
Colab par Tesseract system-level software hai, isliye pehle `apt-get` se install karna hoga.

Input PDF → Docling configuration → Tesseract OCR + layout analysis + table detection → DoclingDocument → JSON / Markdown / Text / CSV / Images

In [ ]:
# Colab ki Linux machine par Tesseract OCR engine install kar rahe hain (system software)
# -y matlab automatically "yes" confirm ho jaye, koi prompt na aaye
!apt-get install -y tesseract-ocr

# Ab Python packages install kar rahe hain: docling, langchain-docling, tabulate
# -q matlab quiet mode, taake zyada output screen par na aaye
!pip install docling langchain-docling tabulate -q

In [ ]:
# Check kar rahe hain ke Tesseract sahi install hua hai aur uska path kya hai
!which tesseract
!tesseract --version

In [ ]:
# Google Colab se apni PDF file upload karne ke liye
# Ye button dikhayega jahan se aap apne computer se PDF choose kar sakte hain
from google.colab import files

uploaded = files.upload()   # upload dialog khulega, PDF select karein
pdf_filename = list(uploaded.keys())[0]   # jo file upload hui uska naam le rahe hain
print("Uploaded file:", pdf_filename)

In [ ]:
import json
from pathlib import Path

# json module data ko JSON format mein save karne ke liye
# Path module file/folder paths ko aasani se handle karne ke liye

In [ ]:
from docling.datamodel.base_models import InputFormat

# InputFormat wo class hai jisse Docling ko batate hain ke
# input file ka type kya hai (yahan PDF)

In [ ]:
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,       # PDF parsing ki saari settings iske andar hoti hain
    TableStructureOptions,    # table detect/extract karne ki settings
    TesseractCliOcrOptions,   # Tesseract OCR engine ki settings (command-line wala)
)

In [ ]:
from docling.document_converter import (
    DocumentConverter,   # main converter/parser class
    PdfFormatOption,      # PDF ke liye specific options wrap karne ke liye
)

In [ ]:
from docling_core.types.doc import (
    ImageRefMode,   # markdown mein images ko kaise reference karna hai (embed ya link)
    PictureItem,    # document ke andar picture/image elements identify karne ke liye
    TableItem,      # document ke andar table elements identify karne ke liye
)

In [ ]:
# Colab ka file system Windows jaisa nahi hota,
# isliye path "/content/..." se shuru hoti hai

PDF_PATH = Path("/content/" + pdf_filename)   # jo PDF upload ki usi ka path

OUTPUT_DIR = Path("/content/docling_parsed_output")   # output files yahan save hongi

PAGE_IMAGE_DIR = OUTPUT_DIR / "page_images"        # har page ki image yahan
PICTURE_DIR = OUTPUT_DIR / "extracted_pictures"    # PDF ke andar ki pictures yahan
TABLE_DIR = OUTPUT_DIR / "extracted_tables"        # tables (csv/md/png) yahan

# mkdir(parents=True) matlab agar parent folder na ho to wo bhi bana do
# exist_ok=True matlab agar folder pehle se ho to error na de
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAGE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PICTURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Agar PDF file maujood nahi to error raise karo, taake pata chal jaye
if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

In [ ]:
# PdfPipelineOptions ek object hai jisme hum saari
# PDF-processing settings (OCR, table extraction, images) set karenge
pipeline_options = PdfPipelineOptions()

In [ ]:
# OCR ko enable kar rahe hain
# do_ocr = True matlab Docling scanned/image-based text ko bhi padhega
pipeline_options.do_ocr = True

In [ ]:
# Yahan hum OCR engine set kar rahe hain: Tesseract
# Colab par apt-get se install hone ke baad Tesseract system PATH mein
# automatically aa jata hai, isliye hardcoded Windows path (tesseract_cmd)
# dene ki koi zaroorat nahi — bas lang aur force_full_page_ocr set karein

pipeline_options.ocr_options = TesseractCliOcrOptions(
    lang=["eng"],   # sirf English language OCR ke liye

    # force_full_page_ocr=False matlab:
    # jahan text already selectable hai wahan OCR nahi karega,
    # sirf image/scanned parts par hi OCR chalega (fast + accurate)
    force_full_page_ocr=False,
)

In [ ]:
# Table structure detection enable kar rahe hain
# yani PDF ke andar jo tables hain unko pehchan kar extract karega
pipeline_options.do_table_structure = True

In [ ]:
# do_cell_matching=True matlab table ke har cell ko
# uski asal row/column position ke sath sahi match karega
pipeline_options.table_structure_options = TableStructureOptions(
    do_cell_matching=True,
)

In [ ]:
# Page aur picture images ko preserve/generate karne ki settings

pipeline_options.images_scale = 2.0            # image resolution 2x rakho (behtar quality)
pipeline_options.generate_page_images = True    # har page ki poori image banao
pipeline_options.generate_picture_images = True # PDF ke andar ki har picture bhi separately save karo

In [ ]:
# Ab main converter object bana rahe hain
# format_options mein bata rahe hain ke PDF files ke liye
# hamari upar wali pipeline_options use karo

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

In [ ]:
print("Parsing PDF with Docling (Tesseract OCR)...")

# Yahan asal mein PDF parse ho rahi hai
# ye line sabse zyada time lene wali hai (OCR + table detection ho raha hai)
conversion_result = converter.convert(PDF_PATH)

In [ ]:
# conversion_result.document mein poora parsed document (structured data) hai
docling_document = conversion_result.document

print("PDF parsed successfully.")
print("Total pages:", len(docling_document.pages))       # kitne pages parse hue
print("Total tables:", len(docling_document.tables))     # kitne tables mile
print("Total pictures:", len(docling_document.pictures)) # kitni pictures mili

In [ ]:
json_output_path = OUTPUT_DIR / "docling_document.json"
# JSON file ka path define kar rahe hain

In [ ]:
# Poora document JSON format mein save kar rahe hain
# indent=2 matlab readable/pretty JSON
# ensure_ascii=False matlab Urdu/Arabic jaise characters bhi sahi save honge

json_output_path.write_text(
    json.dumps(
        docling_document.export_to_dict(),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("JSON saved:", json_output_path)

In [ ]:
markdown_output_path = OUTPUT_DIR / "rag_ready_document.md"

# Document ko Markdown format mein save kar rahe hain
# ImageRefMode.REFERENCED matlab images ko separate files ki tarah link karega,
# poora image data markdown ke andar embed nahi karega (file halki rehti hai)

docling_document.save_as_markdown(
    markdown_output_path,
    image_mode=ImageRefMode.REFERENCED,
)

print("Markdown saved:", markdown_output_path)

In [ ]:
text_output_path = OUTPUT_DIR / "extracted_text.txt"

# strict_text=True matlab sirf plain text nikalo,
# formatting (headings, tables ka structure) skip kar do
text_output_path.write_text(
    docling_document.export_to_markdown(strict_text=True),
    encoding="utf-8",
)

print("Plain text saved:", text_output_path)

In [ ]:
# Har page ki image ko PNG file ki tarah save kar rahe hain
for page_number, page in docling_document.pages.items():

    if page.image is None:      # agar page ki image generate nahi hui to skip karo
        continue

    page_image_path = (
        PAGE_IMAGE_DIR
        / f"page_{page.page_no:03d}.png"   # jaise page_001.png, page_002.png
    )

    page.image.pil_image.save(
        page_image_path,
        format="PNG",
    )

print("Page images saved:", PAGE_IMAGE_DIR)

In [ ]:
# Document mein jitne bhi tables mile unko loop kar rahe hain
for table_index, table in enumerate(
    docling_document.tables,
    start=1,   # counting 1 se shuru hogi (table_001, table_002...)
):
    # Table ko pandas DataFrame mein convert kar rahe hain
    table_df = table.export_to_dataframe(
        doc=docling_document
    )

    csv_path = TABLE_DIR / f"table_{table_index:03d}.csv"
    markdown_path = TABLE_DIR / f"table_{table_index:03d}.md"
    image_path = TABLE_DIR / f"table_{table_index:03d}.png"

    # Table ko CSV format mein save kar rahe hain
    table_df.to_csv(
        csv_path,
        index=False,   # row numbers save nahi karne
        encoding="utf-8",
    )

    # Table ko Markdown table format mein bhi save kar rahe hain
    markdown_path.write_text(
        table_df.to_markdown(index=False),
        encoding="utf-8",
    )

    # Table ka visual/image version bhi nikal rahe hain
    table_image = table.get_image(docling_document)

    if table_image is not None:
        table_image.save(image_path, format="PNG")

print("Tables saved:", TABLE_DIR)

In [ ]:
picture_counter = 0

# iterate_items() poore document ke elements (text, table, picture...) ek ek karke deta hai
for element, _level in docling_document.iterate_items():

    if not isinstance(element, PictureItem):   # agar element picture nahi hai to skip
        continue

    picture_counter += 1

    picture_path = (
        PICTURE_DIR
        / f"picture_{picture_counter:03d}.png"
    )

    picture_image = element.get_image(docling_document)   # asal image data nikalo

    if picture_image is not None:
        picture_image.save(
            picture_path,
            format="PNG",
        )

print("Pictures saved:", PICTURE_DIR)

In [ ]:
# Sab kuch OUTPUT_DIR (/content/docling_parsed_output) mein save ho chuka hai
# Isko zip karke apne computer par download kar sakte hain:

import shutil
shutil.make_archive("/content/docling_parsed_output", "zip", OUTPUT_DIR)

from google.colab import files
files.download("/content/docling_parsed_output.zip")